# DRACO with the ScreamingFace SDK

Build an open-ended research fusion, execute every model through URL4, and compare its weighted
DRACO rubric score with the same panel members scored individually.

The saved run uses two bundled DRACO-shaped cases and an in-process URL4 node with deterministic
model routes. It validates the complete ScreamingFace request and response contract without setup,
credentials, or a provider-quality claim.

The final section identifies the remaining production boundary: the configured URL4 model routes
must execute the capabilities and parameters that ScreamingFace emits.

## 1 · Import and configure

In [1]:
import screamingface as sf

# Optional: send every model and judge call to an HTTP URL4 engine.
# sf.config("http://127.0.0.1:4404")  # first run ./scripts/dev-url4.sh
# sf.config("https://url4.example")

## 2 · Define the research behavior

These prompts belong to the experiment, not the DRACO dataset adapter. `$question` is resolved for
each case. The reducer additionally receives the labeled `$panel_answers` produced by URL4.

In [2]:
DRACO_PANEL_PROMPT = """
You are answering a research-quality prompt. Provide a thorough, well-reasoned answer
in prose. Address every aspect, preserve specific facts and sources, and use clear
structure.

Research prompt:
$question
""".strip()

DRACO_REDUCER_PROMPT = """
Produce one comprehensive answer to the research prompt by combining the strongest
facts, arguments, and citations from every labeled panel answer. Resolve disagreements
in favor of the more specific and better-supported claim. Return only the unified prose
answer.

Research prompt:
$question

Panel answers:
$panel_answers
""".strip()

## 3 · Compose the fusion

In [3]:
fusion = sf.Fusion(
    "draco-frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini-cli/gemini-2.5-pro",
        "anthropic/claude-sonnet-4-6",
    ],
    prompt=DRACO_PANEL_PROMPT,
    tools=["web_search"],
    reducer=sf.ModelReducer(
        model="codex/gpt-5.5",
        prompt=DRACO_REDUCER_PROMPT,
        params={"temperature": 0.0, "max_tokens": 8192},
    ),
)
fusion

Role,Model
Model,codex/gpt-5.5
Model,gemini-cli/gemini-2.5-pro
Model,anthropic/claude-sonnet-4-6


The shareable fusion URL4 contains the unresolved panel and reducer graph. It does
not contain the DRACO dataset or rubric, and displaying it executes nothing.

In [4]:
fusion.url4

"(panel_1=/codex/gpt-5.5?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_2=/gemini/2.5?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_3=/claude/sonnet-4.6?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_answers={panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2

## 4 · Evaluate

For each case, ScreamingFace sends one fusion expression to `/v1`. After receiving the panel and
synthesized answers, the DRACO grader sends one additional URL4 model request for every
answer × rubric criterion × judge pass. No SDK code calls AI Gateway or a provider directly.

In [5]:
run = fusion.evaluate("draco", first=2, seed=0)
run

Run(benchmark='DRACO-shaped synthetic research fixture', dataset_source='synthetic-draco-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(panel_1=/codex/gpt-5.5?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_2=/gemini/2.5?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_3=/claude/sonnet-4.6?tools=web_search&q=()!'You are answering a research-quality prompt. Provide a thorough, well-reasoned answer\nin prose. Address every aspect, preserve specific facts and sources, and use clear\nstructure.\n\nResearch prompt:\n$question', panel_answers={panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'}, fusion_answer=/codex/gpt-5.5?temperature=0.0&max_tokens=8192&q=()!'Produce one comprehensive answer to the research prompt by combining the strongest\nfacts, arguments, and citations from every labeled panel answer. Resolve disagreements\nin favor of the more specific and better-supported claim. Return only the unified prose\nanswer.\n\nResearch prompt:\n$question\n\nPanel answers:\n$panel_answers', {schema: 'screamingface.fusion-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3', reducer: 'model', reducer_model: 'codex/gpt-5.5', answer: '$fusion_answer'})", sample_size=2, seed=0, score=100.0, baseline=33.3, gain=66.7, cost_usd=0.0, engine='mock', fusion_name='draco-frontier-trio', reducer='model', tie_breaker=None, incomplete=0, profiles=(), pricing_source='engine response does not yet report usage', pricing_as_of='n/a', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=33.3, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('factual_accuracy', 33.3), ('normalized_score', 33.3), ('pass_rate', 50.0), ('verdict_coverage', 100.0))), ModelResult(model='gemini-cli/gemini-2.5-pro', score=33.3, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('factual_accuracy', 33.3), ('normalized_score', 33.3), ('pass_rate', 50.0), ('verdict_coverage', 100.0))), ModelResult(model='anthropic/claude-sonnet-4-6', score=33.3, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('factual_accuracy', 33.3), ('normalized_score', 33.3), ('pass_rate', 50.0), ('verdict_coverage', 100.0)))), failures=(), primary_metric='normalized_score', metrics=(('factual_accuracy', 100.0), ('normalized_score', 100.0), ('pass_rate', 100.0), ('verdict_coverage', 100.0)))

## 5 · Read the comparison

In [6]:
{
    "primary_metric": run.primary_metric,
    "fusion_score": run.score,
    "best_member": run.baseline,
    "gain": run.gain,
    "rubric_metrics": dict(run.metrics),
}

{'primary_metric': 'normalized_score',
 'fusion_score': 100.0,
 'best_member': 33.3,
 'gain': 66.7,
 'rubric_metrics': {'factual_accuracy': 100.0,
  'normalized_score': 100.0,
  'pass_rate': 100.0,
  'verdict_coverage': 100.0}}

`normalized_score` is DRACO's weighted score: positive criteria add their weights,
MET negative criteria subtract their weights, and the result is divided by total positive weight
and clamped to 0–100. `gain` compares the synthesis with the best panel member on the same cases
and judge protocol.

## 6 · What the production URL4 engine must handle

The bundled in-process URL4 node validates the request and response shapes deterministically.
Replacing it with a production HTTP URL4 engine requires the engine to:

- accept each complete expression at `GET /v1?q=<url4 expression>` and execute its dependency graph;
- dispatch every `/provider/model` node to the corresponding production model route;
- translate `tools=web_search` on panel nodes into each provider's native search capability;
- preserve judge-request URL4 intent as the system message and context as the user message;
- forward `temperature=0.2`, `reasoning=low`, and `max_tokens=4096` for judge calls;
- run repeated judge expressions as independent samples rather than collapsing or caching them;
- keep research tools disabled for judge calls, which only grade supplied responses;
- contact AI Gateway internally—ScreamingFace never contacts it or a provider directly;
- return panel/fusion outputs and raw judge JSON in the response shapes demonstrated above; and
- eventually return usage, cost, failure, retry, search, and citation telemetry.

Until those production routes exist, the saved result demonstrates the complete HTTP contract but
makes no claim about provider quality or production DRACO scores.